# Day 11 — Exceptions & Modules
### Python for Data Science · Module 1 · Topic 1.10

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Reading a traceback | 15 min |
| 2 | `try` / `except` / `else` / `finally` | 25 min |
| 3 | Raising your own exceptions | 20 min |
| 4 | Modules and packages | 25 min |
| 5 | Mini build: a CSV loader that survives bad data | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **The principle behind all of it:** an exception you did **not** expect should stop the
> program loudly, so you find out about it. An exception you **did** expect should be
> handled quietly, so the program carries on. Most bad error handling comes from confusing
> the two — usually by catching everything and hiding a real bug for months.

---
## 0. You have already met eight exception types

| Exception | First seen | Cause |
|---|---|---|
| `ValueError` | Day 1 | `int("abc")` |
| `TypeError` | Day 1 | `"10" + 5` |
| `NameError` | Day 3 | a variable that is not there |
| `UnboundLocalError` | Day 3 | `count = count + 1` inside a function |
| `IndexError` | Day 4 | `marks[99]` |
| `KeyError` | Day 5 | `d["missing"]` |
| `AttributeError` | Day 7 | a missing `self.name` |
| `FileNotFoundError` | Day 9 | a file that is not there |

So far each one has meant *your program stops and you go back to fix the code*. That is
right during development. But a program that reads a file the user chose, or takes input
you cannot control, has to keep working when something goes wrong.

---
# 1. Reading a traceback

```
Traceback (most recent call last):
  File "analysis.py", line 12, in <module>
    result = average(marks)
  File "analysis.py", line 7, in average
    return total / len(values)
ZeroDivisionError: division by zero
```

**Start at the BOTTOM.** The last line names the error type and describes it. The frames
above are printed oldest first, so the line that actually failed is just above the error;
read upwards only if you need to know who called it.

`"most recent call last"` is an instruction about the ordering, not a description of the
problem. Students who read tracebacks top-down waste a great deal of time on files they
never wrote.

In [ ]:
def average(values):
    total = sum(values)
    return total / len(values)

def report(marks):
    return f"Average: {average(marks)}"

# Run this and read the traceback from the BOTTOM upwards
report([])

## 1.1 Exceptions are classes — and they inherit

```
        BaseException
              │
          Exception
        ┌─────┼─────┐
  ValueError  LookupError  OSError
                 │            │
        ┌────────┴──┐    FileNotFoundError
    KeyError   IndexError
```

Catching a parent catches every child — which is useful when you genuinely mean
*"any lookup failure"*, and dangerous when you write `except Exception` and accidentally
swallow bugs you never considered.

> **This is Day 8, applied.** An exception is an object built from a class, and
> `except ValueError` is really an `isinstance` check.

In [ ]:
print(issubclass(KeyError, LookupError))       # True
print(issubclass(IndexError, LookupError))     # True
print(issubclass(FileNotFoundError, OSError))  # True
print(issubclass(ValueError, Exception))       # True

# So one block can catch both:
for bad in [lambda: {"a": 1}["z"], lambda: [1, 2][99]]:
    try:
        bad()
    except LookupError as e:
        print(f"caught {type(e).__name__}: {e}")

---
# 2. try / except

## 2.1 The basic shape

In [12]:
raw = "sai"

try:
    marks = int(raw)
except ValueError:
    print(f"{raw} was not a number, you need to enter a interger or float for this")
    marks = 0

print("marks =", marks)

sai was not a number, you need to enter a interger or float for this
marks = 0


In [3]:
raw = "sai"
marks = int(raw)
print("marks =", marks)

ValueError: invalid literal for int() with base 10: 'sai'

In [ ]:
# Catching several types, and seeing the message
for raw in ["88", "abc", None]:
    try:
        print(int(raw))
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")

### ⚠️ Order matters — specific first, general last

In [ ]:
# WRONG - the general block catches everything, so the specific one never runs
try:
    int("abc")
except Exception:
    print("general block ran")
except ValueError:
    print("this line is unreachable")

In [ ]:
# RIGHT - specific types first
try:
    int("abc")
except ValueError:
    print("specific block ran")
except Exception:
    print("general fallback")

Python tries each `except` block in order and takes the first that matches. Because
`ValueError` inherits from `Exception`, a general block written first swallows everything
below it — **exactly the elif-ordering rule from Day 2.**

## 2.2 The full form: try / except / else / finally

In [ ]:
def demo(path):
    try:
        f = open(path, encoding="utf-8")
    except FileNotFoundError:
        print("  except : no such file")
    else:
        print("  else   : opened fine")
        f.close()
    finally:
        print("  finally: always runs")

open("exists.txt", "w").write("hi")

print("with a file that exists:")
demo("exists.txt")

print("\nwith a file that does not:")
demo("nope.txt")

| | No error | Error caught |
|---|---|---|
| `try` | runs fully | stops at the error |
| `except` | skipped | runs |
| `else` | runs | skipped |
| `finally` | **runs** | **runs** |

`finally` runs even if the `try` block executes a `return`, or raises something nothing
caught. That is what makes it the right place for cleanup.

In [ ]:
def f():
    try:
        return "returned from try"
    finally:
        print("finally still ran, even with a return")

print(f())

> **You already use `finally` every day.** Day 9's `with open(...)` is built on exactly this
> mechanism — the file closes whether the block succeeds, raises, or returns early. When a
> `with` statement exists for something, prefer it.

**Keep the try block as SHORT as possible** — only the line that might fail. Anything else
in there can raise an exception you did not mean to catch.

## 2.3 The two ways error handling goes wrong

In [20]:
# ⚠️ ANTI-PATTERN 1: the bare except
def process(data):
    return int(data["mark"])

try:
    process({"mark": "sai"})
    print("successful")          # note the typo in the key
except:                             # catches ABSOLUTELY everything
    print("something went wrong")   # ...but what? and where?

# It hides your typos, your NameErrors, and even Ctrl+C.
# If you truly need a catch-all, write  except Exception  - at least that
# leaves KeyboardInterrupt alone so the program can still be stopped.

something went wrong


In [22]:
def process(data):
    return int(data["mark"])

process({"mark": "sai"})

ValueError: invalid literal for int() with base 10: 'sai'

In [ ]:
# ⚠️ ANTI-PATTERN 2: the silent pass
rows = [{"mark": "88"}, {"mark": "abc"}, {"mark": "71"}]

total = 0
for row in rows:
    try:
        total += int(row["mark"])
    except ValueError:
        pass                        # the bad row vanishes without trace

print("total:", total)              # 159 - but nobody knows a row was skipped

In [ ]:
# THE FIX: if you deliberately ignore something, COUNT it
total, skipped = 0, 0
for row in rows:
    try:
        total += int(row["mark"])
    except ValueError:
        skipped += 1                # visible, reportable

print(f"total: {total}, skipped: {skipped} bad rows")

> ### Three questions to ask before you write `except`
>
> 1. **Which exception exactly?** — name it
> 2. **What should happen instead?** — a default, a retry, a message
> 3. **Would I want to know?** — if yes, log it or re-raise
>
> If you cannot answer all three, you do not yet know what you are handling.

---
# 3. Raising your own

## 3.1 `raise` — reporting a problem at its source

In [24]:
# ⚠️ The anti-pattern: returning an error message
def set_age_bad(age):
    if age < 0:
        return "age cannot be negative"
    return age

result = set_age_bad(-5)
# try:
#     print(result + 10)
# except TypeError as e:
#     print("TypeError:", e)
#     print("  -> the failure surfaced FAR from the real problem")

In [32]:
# The fix: raise, and stop it at the source
def set_age(age):
    if age < 0:
        raise ValueError(f"age cannot be negative, got {age}")
    return age

try:
    set_age(-5)
except ValueError as e:
    print("ValueError:", e)

ValueError: age cannot be negative, got -5


**Pick the built-in type that already describes the problem:**

| Type | Use it for |
|---|---|
| `ValueError` | right type, unacceptable value |
| `TypeError` | wrong type altogether |
| `KeyError` | a missing key |
| `FileNotFoundError` | a missing file |

Always include a message saying **what was wrong and what was expected**.
*"Invalid input"* helps nobody.

## 3.2 Custom exceptions

In [ ]:
class InvalidMarkError(Exception):
    """A mark outside the range 0-100."""


def record(mark):
    if not 0 <= mark <= 100:
        raise InvalidMarkError(f"got {mark}, expected 0-100")
    return mark


try:
    record(150)
except InvalidMarkError as e:
    print("InvalidMarkError:", e)

print("is it an Exception?", issubclass(InvalidMarkError, Exception))

That is the whole definition — a name, a parent, and a docstring. No methods needed.

**Why bother, instead of `ValueError`?**

- Callers can catch **your** error without also catching every unrelated `ValueError`
- The name documents the rule that was broken
- You can group them: give your project one base error and inherit the rest from it

In [ ]:
# ⚠️ A class that does NOT inherit from Exception cannot be raised
class NotAnException:
    pass

try:
    raise NotAnException("nope")
except TypeError as e:
    print("TypeError:", e)

## 3.3 Re-raising

In [ ]:
def load(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"  [log] missing file: {path}")
        raise                 # bare raise re-throws the SAME error

try:
    load("nope.txt")
except FileNotFoundError as e:
    print("caller decided what to do:", type(e).__name__)

A bare `raise` inside an `except` block re-throws what you just caught, keeping the original
traceback intact. Use it when you want to record that something happened but still let the
caller decide what to do about it.

> **Logging is not handling.** Printing the error does not mean you have dealt with it.

---
# 4. Modules and packages

## 4.1 Four ways to import

| You write | You then use | When |
|---|---|---|
| `import math` | `math.sqrt(9)` | the default — the source stays visible |
| `import numpy as np` | `np.array(...)` | long names, or a community convention |
| `from math import sqrt` | `sqrt(9)` | one or two names you use constantly |
| `from math import *` | `sqrt(9)` | **never** |

In [ ]:
import math
from math import sqrt
import statistics as stats

print(math.sqrt(9), math.pi)
print(sqrt(16))
print(stats.mean([1, 2, 3, 4]))

### ⚠️ Why `from x import *` is banned everywhere

Every name lands in your namespace at once, silently shadowing anything with the same name —
including your own variables. You lose all trace of where a function came from.

In [ ]:
# A demonstration of the danger
def sqrt(x):
    return "my own sqrt"

print(sqrt(9))            # my own version

from math import *        # silently replaces it

print(sqrt(9))            # 3.0 - your function is gone, with no warning

### The conventions you will see everywhere

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
```

Use these exact aliases. Every tutorial, answer and colleague assumes them — and you will
start using the first two tomorrow.

## 4.2 The standard library — already installed, already tested

| Module | Useful names | You have already met it |
|---|---|---|
| `math` | `sqrt`, `pi`, `floor`, `isclose` | Day 1's float comparison |
| `random` | `randint`, `choice`, `shuffle` | Day 2's guessing game |
| `datetime` | dates, times, differences | timestamps |
| `csv` | `reader`, `DictReader` | Day 9 |
| `json` | `loads`, `dumps` | web APIs, config files |
| `collections` | `Counter`, `defaultdict` | Days 5 and 6 |
| `statistics` | `mean`, `median`, `stdev` | a preview of Module 2 |
| `pathlib` | paths, existence | Day 9 |

In [ ]:
import random, statistics, json
from collections import Counter
from datetime import datetime

random.seed(42)
marks = [random.randint(40, 100) for _ in range(8)]

print("marks   :", marks)
print("mean    :", round(statistics.mean(marks), 1))
print("median  :", statistics.median(marks))
print("commonest first digit:", Counter(str(m)[0] for m in marks).most_common(1))
print("as JSON :", json.dumps({"marks": marks})[:40], "...")
print("now     :", datetime.now().strftime("%Y-%m-%d"))

> Before you write a helper function, spend two minutes checking whether the standard
> library already has it. It will be better tested than yours, and every Python programmer
> will recognise it.

## 4.3 Your own modules

A **module** is just a `.py` file. The filename becomes the module name.

In [ ]:
# Normally you would create helpers.py in your editor. We build it from here
# so the whole lesson stays inside one notebook.
lines = [
    'def clean(name):',
    '    """Strip spaces and title-case a name."""',
    '    return name.strip().title()',
    '',
    'def average(values):',
    '    """Mean of a list, or 0 for an empty list."""',
    '    return sum(values) / len(values) if values else 0',
    '',
    'if __name__ == "__main__":',
    '    print("run directly - this does NOT print on import")',
]

with open("helpers.py", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

import helpers

print(helpers.clean("  ravi kumar "))
print(helpers.average([88, 71, 64]))
print(helpers.clean.__doc__)

Notice the `if __name__ == "__main__":` block did **not** print. Code under that line runs
when the file is executed directly, but not when it is imported — which is how one file can
be both a reusable module and a runnable script.

A **package** is a folder of modules containing an `__init__.py` file:

```
myproject/
    __init__.py       marks it as a package (can be empty)
    loading.py
    cleaning.py
```

```python
from myproject import cleaning
```

This is how pandas and numpy are organised internally.

**Installing what is not built in:** `pip install pandas` in a terminal, or `!pip install pandas`
inside a Colab cell. Colab already has numpy, pandas and matplotlib.

---
# 5. Putting it together — a CSV loader that survives bad data

In [ ]:
import csv


class InvalidMarkError(Exception):
    """A mark that is not a number between 0 and 100."""


def parse_mark(raw):
    try:
        mark = int(raw)                     # may raise ValueError
    except ValueError:
        raise InvalidMarkError(f"not a number: {raw!r}")
    if not 0 <= mark <= 100:
        raise InvalidMarkError(f"out of range: {mark}")
    return mark


def load(path):
    """Load rows, skipping bad ones but REPORTING them."""
    good, bad = [], []
    try:
        with open(path, newline="", encoding="utf-8") as f:
            for n, row in enumerate(csv.DictReader(f), start=2):
                try:
                    row["mark"] = parse_mark(row["mark"])
                    good.append(row)
                except InvalidMarkError as e:
                    bad.append(f"line {n}: {e}")      # counted, not hidden
    except FileNotFoundError:
        raise                                        # caller decides
    return good, bad

In [ ]:
# Build a file with two deliberately bad rows
with open("marks.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "mark"])
    w.writerows([["Ravi", 88], ["Sara", "abc"], ["Amit", 150], ["Neha", 71]])

good, bad = load("marks.csv")

print("loaded :", [(r["name"], r["mark"]) for r in good])
print("bad    :")
for b in bad:
    print("   ", b)

# And a missing file goes UP to the caller
try:
    load("missing.csv")
except FileNotFoundError:
    print("\nmissing file propagated to the caller, as it should")

- **Custom exception** — names the rule that was broken
- **Narrow try** — only the line that can fail
- **Translate** — a `ValueError` becomes a meaningful domain error
- **Per-row catch** — one bad row does not stop the whole load
- **Never silent** — bad rows are collected and returned
- **`enumerate(start=2)`** — line numbers a human can actually use
- **Bare `raise`** — a missing file is not ours to handle

> A bad row is **expected**, so it is handled. A missing file is **not**, so it goes up to
> the caller. That distinction is the whole session.

---
# 6. Recap — the twelve things to remember

1. Read a traceback from the **bottom** — that line names the error.
2. Exceptions are classes, so catching a parent catches its children.
3. Catch the **specific** exception you expected, never everything.
4. Specific `except` blocks first, general ones last — like `elif`.
5. `else` runs only if nothing failed; `finally` runs always.
6. Never write a bare `except:` — it catches typos and Ctrl+C.
7. Never `except: pass` — count it or log it, but do not hide it.
8. `raise ValueError("what and why")` beats returning a message.
9. A custom exception is a class inheriting from `Exception`.
10. A bare `raise` re-throws what you caught, traceback intact.
11. `import math` is the default; `from x import *` is never right.
12. A module is a `.py` file; a package is a folder with `__init__.py`.

---

### 📝 Now open **`Day11_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Add proper exception handling to your Day 9 CSV functions.
- Write a custom exception for one rule in your own code.
- Split a notebook into a `helpers.py` module and import it.

### Next class — Topic 1.11: NumPy
Array creation and indexing: `np.array`, `arange`, `zeros` and `ones`, slicing, reshaping.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*